# Loading the Libraries

In [22]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from scipy.sparse import csr_matrix

# Use a consistent style for all charts in this notebook.
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")

print("\n✓ Environment check passed")

Python: 3.14.2
NumPy: 2.5.3
pandas: 3.0.5

✓ Environment check passed


# Loading the data

In [23]:
# Load the dataset
df = pd.read_csv('../original_data/DataSet_AIvsHuman.csv')
df.head()


,id,prompt,content,type,source,label,topic,word_count,char_count,ai_model,language,complexity_score,is_multiline_code
0,1,Write a string manipulation solution in python,def string_manipulation():\n # TODO: implem...,code,blog,human,string manipulation,9,84,NaN,python,3,True
1,2,Explain health in simple terms,Experts recommend focusing on health to improv...,text,generated,ai,health,29,211,Claude,en,3,False
2,3,Summarize recent developments in finance,The development of finance technologies has ac...,text,generated,ai,finance,56,419,GPT-4,en,5,False
3,4,Summarize recent developments in finance,Recent studies show that finance can have sign...,text,generated,ai,finance,42,298,Claude,en,10,False
4,5,Provide a graph traversal algorithm in cpp,vector<int> arr; // graph traversal example\nf...,code,generated,ai,graph traversal,11,98,Gemini,cpp,10,True


# Splitting the data into text only

In [24]:

df_text = df[df["type"].str.lower() == "text"]
df_text.head()
print("rows, columns:", df_text.shape)

#df_text.select_dtypes("number").hist(figsize=(10, 6), bins=30, edgecolor="white")
#plt.tight_layout()
#plt.show()
df_text.info()

rows, columns: (10004, 13)
<class 'pandas.DataFrame'>
Index: 10004 entries, 1 to 19992
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 10004 non-null  int64
 1   prompt             10004 non-null  str  
 2   content            10004 non-null  str  
 3   type               10004 non-null  str  
 4   source             10004 non-null  str  
 5   label              10004 non-null  str  
 6   topic              10004 non-null  str  
 7   word_count         10004 non-null  int64
 8   char_count         10004 non-null  int64
 9   ai_model           5067 non-null   str  
 10  language           10004 non-null  str  
 11  complexity_score   10004 non-null  int64
 12  is_multiline_code  10004 non-null  bool 
dtypes: bool(1), int64(4), str(8)
memory usage: 4.6 MB


## Target Leakage
> removing all numeric columns

In [25]:
target = "label"

leaky_cols = ["source", "ai_model", "language", "type", "is_multiline_code", "prompt", "topic", "word_count", "char_count", "complexity_score"]
id_cols = ["id"]

X = df_text.drop(columns=[target] + leaky_cols + id_cols)
y = df_text[target]
print("Features used:", X.columns.tolist())

X.head()
print("rows, columns:", X.shape)

Features used: ['content']
rows, columns: (10004, 1)


# Train / test split

From here on, **anything that learns from the data is fitted on `X_train` only** and then applied to `X_test`.

The test set is looked at **once, at the end**.

In [26]:
# Keep 20% of the rows aside. random_state makes this split reproducible.
# Created from 'leakagefree_data.csv'
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("train:", X_train.shape, " test:", X_test.shape)

# Sections 5-10 change Xtr/Xte. These copies keep X_train/X_test untouched
# for the all-in-one pipeline in section 11.
#Xtr, Xte = X_train.copy(), X_test.copy()

#X_train.to_csv("../created_data/training_data_pipeline.csv", index=False)
#X_test.to_csv("../created_data/testing_data_pipeline.csv", index=False)


train: (8003, 1)  test: (2001, 1)


## TF-IDF 
> https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html

Term Frequency-Inverse Document Frequency (TF-IDF) is a text vectorisation technique that converts documents into numerical feature vectors. The approach assigns high weights to terms that occur frequently within a document but infrequently across the corpus, thereby highlighting words that are more discriminative for classification tasks. TF-IDF is widely used in information retrieval and text classification because it reduces the influence of very common words while emphasising terms that help distinguish between documents.

In [28]:
# Step 1: Createing the TF-IDF vectoriser 
tfidf = TfidfVectorizer(
    analyzer="char",    # Uses character sequences instead of words.
    ngram_range=(3, 5),
    max_features=10000,
    min_df=3,
    sublinear_tf=True
)

# Step 2: Fit only on the training text
X_train_tfidf = tfidf.fit_transform(
    X_train["content"].fillna("")
)

# Step 3: Transform the test data
X_test_tfidf = tfidf.transform(
    X_test["content"].fillna("")
)

# Step 4: Check the result
print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

# 8003/2001 training documents
# 1296/1296 TF-IDF features

pd.DataFrame(
    X_train_tfidf[:5].toarray()
).head()

X_train_tfidf shape: (8003, 1296)
X_test_tfidf shape: (2001, 1296)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,...,1271,1272,1273,1274,1275,1276,1277,1278,1279,1280,1281,1282,1283,1284,1285,1286,1287,1288,1289,1290,1291,1292,1293,1294,1295
0,0.029383,0.029383,0.029383,0.043497,0.043497,0.043497,0.025381,0.025381,0.025381,0.025803,0.025803,0.025803,0.000000,0.000000,0.000000,0.061305,0.043497,0.043497,0.043497,0.043497,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.025381,0.025381,0.025381,0.025381,0.027221,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.050084,0.082547,0.0,0.0,0.0
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.035349,0.035349,0.035349,0.035937,0.035937,0.035937,0.020955,0.020955,0.020955,0.000000,0.000000,0.000000,0.000000,0.000000,0.110932,0.110932,0.110932,0.023211,0.023211,...,0.035349,0.035349,0.035349,0.035349,0.022391,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.045455,0.045455,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0
2,0.028342,0.028342,0.028342,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.052232,0.052232,0.052232,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.027218,0.027218,...,0.000000,0.000000,0.000000,0.000000,0.055103,0.0,0.0,0.047741,0.0,0.050283,0.0,0.0,0.0,0.0,0.0,0.0,0.090247,0.090247,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.020499,0.020499,0.020499,0.000000,0.000000,0.000000,0.034836,0.034836,0.034836,0.000000,0.000000,0.000000,0.000000,0.000000,0.108921,0.108921,0.108921,0.038588,0.038588,...,0.020499,0.020499,0.020499,0.020499,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0
4,0.038265,0.038265,0.038265,0.033456,0.033456,0.033456,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.033174,0.033174,0.033174,0.056645,0.033456,0.033456,0.033456,0.033456,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0


## Week 6
Like previous lectures, we also set `RANDOM_STATE = X`. Some steps (such as splitting the data) involve random choices. Fixing this "seed" makes the random choices the same every time, so everyone running the notebook gets the same results. The number 42 itself is arbitrary; any number works.

## Train a Logistic Regression classifier

In [29]:
RANDOM_STATE = 42

log_reg = make_pipeline(
    StandardScaler(with_mean=False),
    LogisticRegression(random_state=RANDOM_STATE)
)

log_reg.fit(X_train_tfidf, y_train)

log_predictions = log_reg.predict(X_test_tfidf)
log_probabilities = log_reg.predict_proba(X_test_tfidf)[:, 1]


print("First 10 predicted classes:", log_predictions[:10])
print("First 10 probabilities of Pass:", np.round(log_probabilities[:10], 2))
print("Test accuracy:", round(accuracy_score(y_test, log_predictions), 3))

First 10 predicted classes: ['human' 'ai' 'human' 'human' 'ai' 'human' 'human' 'ai' 'ai' 'ai']
First 10 probabilities of Pass: [1. 0. 1. 1. 0. 1. 1. 0. 0. 0.]
Test accuracy: 1.0


In [30]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(X_train_tfidf, y_train)

pred = dummy.predict(X_test_tfidf)

print(
    accuracy_score(y_test, pred)
)

0.4672663668165917


What we have learnt from this model is that our dataset is too easy to predict if it is ai or not to use the id-idf classifier 